# DefectosCafeVerde — DVF1 confirmation seed 2026

Fuser-only confirmation using immutable DVF1 caches. The detector is never loaded or trained; validation is evaluated once after epoch 100 and test remains locked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

relative=Path('experiments/defectoscafeverde-dvf2-v1/DVF2_seed42_result.json')
roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
matches=sorted({p for root in roots if root.exists() for p in root.rglob(relative.name) if p.as_posix().endswith(relative.as_posix())})
if not matches: raise FileNotFoundError(f'DVF2 result tidak ditemukan: {relative}')
digests={sha256(path) for path in matches}
if len(digests)!=1: raise RuntimeError(f'Ditemukan DVF2 result berbeda: {matches}')
preferred=[p for p in matches if '/MyDrive/' in p.as_posix()]
DVF2_RESULT=(preferred or matches)[0]; PROJECT=DVF2_RESULT.parents[2]
DVF1_ROOT=PROJECT/'experiments/defectoscafeverde-dvf1-v1'
DVF1_RESULT=DVF1_ROOT/'DVF1_seed42_result.json'; DVF1_REVIEW=DVF1_ROOT/'DVF1_seed42_review.json'
for path in (DVF1_RESULT,DVF1_REVIEW,DVF2_RESULT,DVF1_ROOT/'train_pair_cache.pt',DVF1_ROOT/'val_pair_cache.pt'):
    if not path.is_file(): raise FileNotFoundError(path)
OUT=PROJECT/'experiments/defectoscafeverde-dvf1-confirmation-v1'; OUT.mkdir(parents=True,exist_ok=True)
SEED=2026
print('PROJECT:',PROJECT); print('SEED:',SEED); print('OUTPUT:',OUT)


In [ ]:
import importlib, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
check=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dvf1_confirmation.py','tests/test_defectoscafeverde_dual_view_fusion.py'],cwd=REPO,text=True,capture_output=True)
print(check.stdout)
if check.returncode: print(check.stderr); raise RuntimeError('DVF1 confirmation tests gagal')


In [ ]:
import time, torch
DEVICE='0' if torch.cuda.is_available() else 'cpu'
LOG=OUT/'DVF1_seed2026_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_dvf1_confirmation','--dvf1-result',str(DVF1_RESULT),'--dvf1-review',str(DVF1_REVIEW),'--dvf2-result',str(DVF2_RESULT),'--output-root',str(OUT),'--seed',str(SEED),'--device',DEVICE,'--authorize-training']
print('START/RESUME DVF1 seed',SEED,'| device=',DEVICE,'| log=',LOG,flush=True)
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    last=''
    while process.poll() is None:
        time.sleep(15)
        lines=LOG.read_text(errors='replace').splitlines()
        status=[line for line in lines if line.startswith(('DVF1 FUSER','REUSE COMPLETE DVF1'))]
        if status and status[-1]!=last: last=status[-1]; print(last,flush=True)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'Seed {SEED} gagal: {process.returncode}')
RESULT=OUT/f'DVF1_seed{SEED}_result.json'; result=json.loads(RESULT.read_text())
assert result['test_images_accessed'] is False
print(json.dumps(result,indent=2,ensure_ascii=False))


In [ ]:
import pandas as pd
metrics=['physical_pair_accuracy','macro_class_accuracy','bottom3_class_accuracy','worst_class_accuracy']
rows=[]
for model in ('PAPER_MAX_CONFIDENCE','DVF1_CANDIDATE','PAIR_ORACLE'):
    rows.append({'model':model,**{metric:result['values'][model][metric] for metric in metrics}})
display(pd.DataFrame(rows).set_index('model').style.format('{:.2%}'))
print('GATES:',result['gates']); print('STATUS:',result['status'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| FUSER TRAINING:',result['fuser_training_executed'],'| TRAINING THIS CALL:',result['training_executed_this_call'],'| TEST:',result['test_images_accessed'])
